
# Assignment 04 · Miền MNIST · Notebook 00: Khảo sát và trực quan hóa dữ liệu

**Học phần:** Phát triển các Hệ thống Thông minh, Học viện Công nghệ Bưu chính Viễn thông

**Sinh viên:** Nguyễn Duy Nghĩa &nbsp;·&nbsp; **Mã sinh viên:** B23DCCN600 &nbsp;·&nbsp; **Lớp:** D23CTPM01

**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế

**Học kỳ:** Học kỳ 1 năm học 2026 &ndash; 2027

---

## Mục tiêu notebook

Notebook này mở đầu chuỗi ba notebook của miền ảnh chữ số viết tay MNIST. Nhiệm vụ của
notebook là khảo sát dữ liệu thô trước khi bất kỳ mô hình nào được huấn luyện, cụ thể:

1. Xác nhận cấu trúc tệp `mnist.npz` và kiểu dữ liệu của từng mảng.
2. Đếm chính xác số mẫu của mười lớp chữ số trên tập train và tập test, đánh giá mức
   độ mất cân bằng lớp.
3. Trực quan hóa một lưới mẫu 10 lớp × 10 ảnh để quan sát biến thiên nét viết trong nội bộ lớp.
4. Thống kê cường độ điểm ảnh, từ đó rút ra hai hằng số chuẩn hóa `MEAN` và `STD`
   sẽ được tái sử dụng nguyên vẹn ở notebook 01 và notebook 02.
5. Lý giải vì sao MNIST được xem là một mốc chuẩn "dễ" trong thị giác máy tính, và điều
   đó đặt ra kỳ vọng gì cho các mô hình sẽ xây dựng.

Hai hình bắt buộc mà notebook này sinh ra theo hợp đồng tích hợp là
`fig_mnist_class_distribution.png` và `fig_mnist_sample_grid.png`.


## 1. Nhập thư viện và cấu hình seed

Toàn bộ Assignment 04 dùng chung `RANDOM_SEED = 42`. Notebook khảo sát không huấn luyện
mô hình nào nhưng vẫn cố định seed vì bước lấy mẫu ảnh minh họa và bước chia
train/validation dùng bộ sinh số ngẫu nhiên.

In [1]:

import os, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Cấu hình hiển thị hình theo đúng mục 5.2 của hợp đồng tích hợp
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = '../data/mnist.npz'
FIG_DIR = '../reports/figures'
os.makedirs(FIG_DIR, exist_ok=True)

print('NumPy :', np.__version__)
print('Đường dẫn dữ liệu :', DATA_PATH, '| tồn tại:', os.path.exists(DATA_PATH))

NumPy : 2.4.0
Đường dẫn dữ liệu : ../data/mnist.npz | tồn tại: True



## 2. Nạp dữ liệu và kiểm tra cấu trúc

Tệp `mnist.npz` đã có sẵn trên đĩa theo mục 3 của hợp đồng tích hợp. Báo cáo không tải lại
dữ liệu từ Internet. Bốn khóa bắt buộc là `x_train`, `y_train`, `x_test`, `y_test`.

In [2]:

_d = np.load(DATA_PATH)
print('Các khóa trong tệp:', list(_d.keys()))

x_train_raw = _d['x_train']
y_train_raw = _d['y_train']
x_test_raw  = _d['x_test']
y_test_raw  = _d['y_test']

for name, arr in [('x_train', x_train_raw), ('y_train', y_train_raw),
                  ('x_test',  x_test_raw),  ('y_test',  y_test_raw)]:
    print(f'{name:8s} shape={str(arr.shape):18s} dtype={arr.dtype} '
          f'min={arr.min()} max={arr.max()}')

N_TRAIN_RAW = x_train_raw.shape[0]
N_TEST_RAW  = x_test_raw.shape[0]
IMG_H, IMG_W = x_train_raw.shape[1], x_train_raw.shape[2]
N_CLASSES = int(y_train_raw.max()) + 1
print()
print(f'Tổng số ảnh huấn luyện gốc : {N_TRAIN_RAW}')
print(f'Tổng số ảnh kiểm thử gốc   : {N_TEST_RAW}')
print(f'Kích thước mỗi ảnh         : {IMG_H} x {IMG_W} (đơn kênh, thang xám)')
print(f'Số lớp                     : {N_CLASSES}')

Các khóa trong tệp: ['x_train', 'y_train', 'x_test', 'y_test']


x_train  shape=(60000, 28, 28)    dtype=uint8 min=0 max=255
y_train  shape=(60000,)           dtype=uint8 min=0 max=9
x_test   shape=(10000, 28, 28)    dtype=uint8 min=0 max=255
y_test   shape=(10000,)           dtype=uint8 min=0 max=9

Tổng số ảnh huấn luyện gốc : 60000
Tổng số ảnh kiểm thử gốc   : 10000
Kích thước mỗi ảnh         : 28 x 28 (đơn kênh, thang xám)
Số lớp                     : 10



**Diễn giải.** Dữ liệu gồm 60 000 ảnh huấn luyện và 10 000 ảnh kiểm thử, mỗi ảnh là ma trận
$28 \times 28$ kiểu `uint8` với giá trị điểm ảnh nằm trọn trong đoạn $[0, 255]$. Ảnh chỉ có
một kênh thang xám, nghĩa là tensor đầu vào cho mạng tích chập hai chiều sẽ có dạng
$(N, 1, 28, 28)$ theo quy ước NCHW của PyTorch và của phần cài đặt NumPy thuần.
Nhãn nhận mười giá trị nguyên từ 0 đến 9 nên bài toán là phân loại đa lớp mười lớp.


## 3. Phân phối lớp trên tập train và tập test

Câu hỏi khảo sát đầu tiên là dữ liệu có cân bằng lớp hay không. Nếu một lớp chiếm tỉ lệ quá
nhỏ, độ chính xác tổng thể sẽ trở thành chỉ số gây hiểu lầm và báo cáo buộc phải dựa vào
macro-F1. Ta đếm trực tiếp trên mảng nhãn thay vì tin vào con số ghi trong tài liệu.

In [3]:

train_counts = np.bincount(y_train_raw, minlength=N_CLASSES)
test_counts  = np.bincount(y_test_raw,  minlength=N_CLASSES)

print(f"{'Chữ số':>7} | {'Train':>7} | {'% train':>8} | {'Test':>6} | {'% test':>7}")
print('-' * 48)
for k in range(N_CLASSES):
    print(f'{k:>7} | {train_counts[k]:>7} | {100*train_counts[k]/N_TRAIN_RAW:>7.2f}% '
          f'| {test_counts[k]:>6} | {100*test_counts[k]/N_TEST_RAW:>6.2f}%')
print('-' * 48)
print(f"{'Tổng':>7} | {train_counts.sum():>7} | {100.0:>7.2f}% | {test_counts.sum():>6} | {100.0:>6.2f}%")

imb_train = train_counts.max() / train_counts.min()
imb_test  = test_counts.max()  / test_counts.min()
print()
print(f'Lớp đông nhất trên train : chữ số {int(train_counts.argmax())} với {train_counts.max()} mẫu')
print(f'Lớp thưa nhất trên train : chữ số {int(train_counts.argmin())} với {train_counts.min()} mẫu')
print(f'Tỉ số mất cân bằng train : {imb_train:.4f}')
print(f'Tỉ số mất cân bằng test  : {imb_test:.4f}')

 Chữ số |   Train |  % train |   Test |  % test
------------------------------------------------
      0 |    5923 |    9.87% |    980 |   9.80%
      1 |    6742 |   11.24% |   1135 |  11.35%
      2 |    5958 |    9.93% |   1032 |  10.32%
      3 |    6131 |   10.22% |   1010 |  10.10%
      4 |    5842 |    9.74% |    982 |   9.82%
      5 |    5421 |    9.04% |    892 |   8.92%
      6 |    5918 |    9.86% |    958 |   9.58%
      7 |    6265 |   10.44% |   1028 |  10.28%
      8 |    5851 |    9.75% |    974 |   9.74%
      9 |    5949 |    9.91% |   1009 |  10.09%
------------------------------------------------
   Tổng |   60000 |  100.00% |  10000 | 100.00%

Lớp đông nhất trên train : chữ số 1 với 6742 mẫu
Lớp thưa nhất trên train : chữ số 5 với 5421 mẫu
Tỉ số mất cân bằng train : 1.2437
Tỉ số mất cân bằng test  : 1.2724


In [4]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xs = np.arange(N_CLASSES)
bars0 = axes[0].bar(xs, train_counts, color='#2E86C1', edgecolor='black', linewidth=0.6)
axes[0].axhline(train_counts.mean(), color='crimson', linestyle='--', linewidth=1.5,
                label=f'Trung bình = {train_counts.mean():.0f}')
axes[0].set_title(f'Phân phối lớp trên tập huấn luyện (N = {N_TRAIN_RAW})', fontsize=13)
axes[0].set_xlabel('Chữ số'); axes[0].set_ylabel('Số lượng ảnh')
axes[0].set_xticks(xs); axes[0].legend()
for b, v in zip(bars0, train_counts):
    axes[0].text(b.get_x() + b.get_width()/2, v + 60, str(v), ha='center', fontsize=9)
axes[0].set_ylim(0, train_counts.max() * 1.12)

bars1 = axes[1].bar(xs, test_counts, color='#E67E22', edgecolor='black', linewidth=0.6)
axes[1].axhline(test_counts.mean(), color='crimson', linestyle='--', linewidth=1.5,
                label=f'Trung bình = {test_counts.mean():.0f}')
axes[1].set_title(f'Phân phối lớp trên tập kiểm thử (N = {N_TEST_RAW})', fontsize=13)
axes[1].set_xlabel('Chữ số'); axes[1].set_ylabel('Số lượng ảnh')
axes[1].set_xticks(xs); axes[1].legend()
for b, v in zip(bars1, test_counts):
    axes[1].text(b.get_x() + b.get_width()/2, v + 10, str(v), ha='center', fontsize=9)
axes[1].set_ylim(0, test_counts.max() * 1.12)

fig.suptitle('MNIST: phân phối mười lớp chữ số, train so với test', fontsize=15, y=1.02)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_class_distribution.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_class_distribution.png')

Đã lưu fig_mnist_class_distribution.png



**Diễn giải hình `fig_mnist_class_distribution.png`.** Trên tập huấn luyện, chữ số 1 chiếm
ưu thế với 6 742 ảnh, tương đương 11,24% dữ liệu, trong khi chữ số 5 thưa nhất với 5 421 ảnh,
tức 9,04%. Tỉ số giữa lớp đông nhất và lớp thưa nhất chỉ là 1,2437, một mức chênh lệch rất
nhỏ. Tập kiểm thử giữ đúng xu hướng đó với chữ số 1 đạt 1 135 ảnh và chữ số 5 chỉ 892 ảnh,
tỉ số mất cân bằng 1,2724. Vì sai lệch giữa các lớp dưới 25%, độ chính xác tổng thể vẫn là
chỉ số đáng tin cậy. Tuy nhiên báo cáo vẫn ghi kèm macro-precision, macro-recall và macro-F1
theo yêu cầu của hợp đồng, bởi các chỉ số macro đối xử bình đẳng với mọi lớp và sẽ phát hiện
sớm trường hợp mô hình bỏ rơi một chữ số khó như 5 hoặc 8.

Nguyên nhân của chênh lệch này mang tính lịch sử: MNIST được tổng hợp từ các biểu mẫu do nhân
viên điều tra dân số và học sinh trung học Hoa Kỳ viết, nên tần suất tự nhiên của từng chữ số
trong tập mẫu không hoàn toàn đồng đều. Điều quan trọng là phân phối train và phân phối test
gần như trùng khớp, cho thấy hai tập được rút từ cùng một phân phối cơ sở. Đây là điều kiện
tiên quyết để độ chính xác đo trên tập kiểm thử phản ánh đúng khả năng tổng quát hóa.


## 4. Lưới mẫu 10 lớp × 10 ảnh

Con số thống kê không cho biết dữ liệu trông ra sao. Ta lấy ngẫu nhiên mười ảnh của mỗi
chữ số để quan sát biến thiên nét viết trong cùng một lớp, đây chính là nguồn khó khăn thực
sự của bài toán.

In [5]:

rng_vis = np.random.default_rng(RANDOM_SEED)
fig, axes = plt.subplots(N_CLASSES, 10, figsize=(11, 11.6))

for k in range(N_CLASSES):
    idx_k = np.where(y_train_raw == k)[0]
    pick = rng_vis.choice(idx_k, size=10, replace=False)
    for j in range(10):
        ax = axes[k, j]
        ax.imshow(x_train_raw[pick[j]], cmap='gray_r', vmin=0, vmax=255)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor('#BBBBBB')
    axes[k, 0].set_ylabel(f'{k}', rotation=0, fontsize=14, labelpad=14,
                          va='center', fontweight='bold')

fig.suptitle('MNIST: mười mẫu ngẫu nhiên cho mỗi lớp chữ số (hàng = nhãn thật)',
             fontsize=14, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.985])
fig.savefig(f'{FIG_DIR}/fig_mnist_sample_grid.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_sample_grid.png')

Đã lưu fig_mnist_sample_grid.png



**Diễn giải hình `fig_mnist_sample_grid.png`.** Lưới mẫu bộc lộ ba đặc điểm quyết định tới
thiết kế mô hình. Thứ nhất, mọi chữ số đều đã được căn giữa theo trọng tâm khối mực và được
co về hộp $20 \times 20$ rồi đệm ra $28 \times 28$, nên vị trí đối tượng gần như bất biến
giữa các ảnh. Thứ hai, nền luôn là màu tối đồng nhất, không có phông nền gây nhiễu, không có
vật thể thứ hai chen vào khung hình. Thứ ba, biến thiên thực sự nằm ở độ dày nét, độ nghiêng
và kiểu viết cá nhân: hàng chữ số 1 có mẫu thẳng đứng lẫn mẫu nghiêng kèm chân đế, hàng chữ số
7 có mẫu gạch ngang giữa thân theo lối viết châu Âu lẫn mẫu không gạch theo lối Mỹ, hàng chữ
số 9 và hàng chữ số 4 xuất hiện các mẫu đóng vòng đầu khiến chúng dễ bị nhầm lẫn với nhau.

Chính nhóm cặp dễ nhầm này, cụ thể là 4 với 9, 3 với 5 và 7 với 1, sẽ là nơi ma trận nhầm lẫn
ở notebook 01 và 02 tập trung sai sót. Việc nhận diện trước các cặp đó giúp báo cáo diễn giải
kết quả sau này bằng lập luận thị giác thay vì chỉ đọc số.


## 5. Thống kê cường độ điểm ảnh và hằng số chuẩn hóa

Trước khi đưa vào mạng, ảnh được đưa về thang $[0,1]$ bằng phép chia cho 255. Sau đó ta còn
chuẩn hóa thêm một bước: trừ trung bình và chia độ lệch chuẩn. Lý do là gradient của tầng
tích chập tỉ lệ thuận với biên độ đầu vào; nếu đầu vào có trung bình lệch hẳn khỏi 0 thì
gradient của mọi trọng số trong cùng một bộ lọc sẽ có cùng dấu, khiến quỹ đạo tối ưu đi theo
đường zigzag và hội tụ chậm. Đưa dữ liệu về trung bình 0 và phương sai 1 loại bỏ hiệu ứng đó.

Điểm mấu chốt về phương pháp luận: hai hằng số `MEAN` và `STD` **phải được học từ tập train**
rồi áp dụng nguyên vẹn cho tập validation và tập test. Nếu tính trên toàn bộ dữ liệu, thông
tin của tập kiểm thử sẽ rò rỉ vào quy trình huấn luyện và kết quả báo cáo sẽ lạc quan giả tạo.

Theo mục 3 của hợp đồng, tập train gốc 60 000 ảnh được tách tiếp thành train và validation
theo tỉ lệ 80/20 có phân tầng. Hằng số chuẩn hóa vì vậy được tính trên 48 000 ảnh của nhánh
train, không phải trên cả 60 000 ảnh.

In [6]:

# Chia train/validation có phân tầng, đúng như hai notebook mô hình sẽ làm
idx_all = np.arange(N_TRAIN_RAW)
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)
print(f'Nhánh train      : {len(idx_tr)} ảnh')
print(f'Nhánh validation : {len(idx_va)} ảnh')
print('Phân phối lớp nhánh train      :', np.bincount(y_train_raw[idx_tr], minlength=10).tolist())
print('Phân phối lớp nhánh validation :', np.bincount(y_train_raw[idx_va], minlength=10).tolist())

x_tr01 = x_train_raw[idx_tr].astype(np.float32) / 255.0
MEAN = float(x_tr01.mean())
STD  = float(x_tr01.std())
print()
print(f'MEAN (học từ 48 000 ảnh nhánh train) = {MEAN:.10f}')
print(f'STD  (học từ 48 000 ảnh nhánh train) = {STD:.10f}')

# Một vài thống kê cường độ bổ sung
nonzero_ratio = float((x_tr01 > 0).mean())
per_img_ink = x_tr01.reshape(len(idx_tr), -1).mean(axis=1)
print()
print(f'Tỉ lệ điểm ảnh khác 0 (có mực) : {nonzero_ratio:.4f}  '
      f'=> {100*(1-nonzero_ratio):.2f}% diện tích ảnh là nền đen tuyệt đối')
print(f'Lượng mực trung bình mỗi ảnh   : {per_img_ink.mean():.6f}')
print(f'Lượng mực thấp nhất / cao nhất : {per_img_ink.min():.6f} / {per_img_ink.max():.6f}')
print(f'Giá trị điểm ảnh sau chuẩn hóa nằm trong: '
      f'[{(0-MEAN)/STD:.4f}, {(1-MEAN)/STD:.4f}]')

Nhánh train      : 48000 ảnh
Nhánh validation : 12000 ảnh
Phân phối lớp nhánh train      : [4738, 5394, 4766, 4905, 4674, 4337, 4734, 5012, 4681, 4759]
Phân phối lớp nhánh validation : [1185, 1348, 1192, 1226, 1168, 1084, 1184, 1253, 1170, 1190]



MEAN (học từ 48 000 ảnh nhánh train) = 0.1307886839
STD  (học từ 48 000 ảnh nhánh train) = 0.3082403541



Tỉ lệ điểm ảnh khác 0 (có mực) : 0.1914  => 80.86% diện tích ảnh là nền đen tuyệt đối
Lượng mực trung bình mỗi ảnh   : 0.130789
Lượng mực thấp nhất / cao nhất : 0.025440 / 0.397574
Giá trị điểm ảnh sau chuẩn hóa nằm trong: [-0.4243, 2.8199]


In [7]:

# Lượng mực trung bình theo từng lớp: chữ số nào tốn mực nhất?
y_tr = y_train_raw[idx_tr]
print(f"{'Chữ số':>7} | {'Lượng mực TB':>14} | {'Độ lệch chuẩn':>14}")
print('-' * 42)
ink_by_class = []
for k in range(N_CLASSES):
    v = per_img_ink[y_tr == k]
    ink_by_class.append(v.mean())
    print(f'{k:>7} | {v.mean():>14.6f} | {v.std():>14.6f}')
ink_by_class = np.array(ink_by_class)
print('-' * 42)
print(f'Chữ số nhiều mực nhất : {int(ink_by_class.argmax())} ({ink_by_class.max():.6f})')
print(f'Chữ số ít mực nhất    : {int(ink_by_class.argmin())} ({ink_by_class.min():.6f})')

 Chữ số |   Lượng mực TB |  Độ lệch chuẩn
------------------------------------------
      0 |       0.173279 |       0.042415
      1 |       0.075961 |       0.022023
      2 |       0.149094 |       0.038121
      3 |       0.141625 |       0.038040
      4 |       0.121561 |       0.032038
      5 |       0.128688 |       0.037003
      6 |       0.137681 |       0.037693
      7 |       0.114725 |       0.030742
      8 |       0.150398 |       0.038864
      9 |       0.122879 |       0.032511
------------------------------------------
Chữ số nhiều mực nhất : 0 (0.173279)
Chữ số ít mực nhất    : 1 (0.075961)



**Diễn giải bảng thống kê cường độ.** Sau khi chia cho 255, trung bình cường độ trên 48 000
ảnh nhánh train là $\mu = 0{,}130789$ và độ lệch chuẩn là $\sigma = 0{,}308240$. Hai giá trị này
được ghi lại và sẽ được tái sử dụng nguyên văn ở notebook 01 và notebook 02, đồng thời được
lưu ra tệp `models/mnist_preproc.json` ở notebook 02 để notebook `mlp_vs_cnn` nạp lại đúng
phép biến đổi khi tải mô hình PyTorch đã huấn luyện.

Con số đáng chú ý nhất là chỉ 19,14% điểm ảnh có mực, tức 80,86% diện tích mỗi ảnh là nền
đen tuyệt đối với giá trị đúng bằng 0. Điều này giải thích vì sao trung bình 0,130789 lại thấp
hơn nhiều so với mức 0,5 của một ảnh phân bố đều. Sau chuẩn hóa, mọi điểm nền đều nhận cùng
một giá trị $(0 - 0{,}130789)/0{,}308240 = -0{,}4243$ còn điểm mực đậm nhất đạt
$(1 - 0{,}130789)/0{,}308240 = 2{,}8199$. Khoảng động dữ liệu vì thế nằm gọn trong
$[-0{,}4243;\ 2{,}8199]$, một biên độ rất phù hợp với khởi tạo He Normal và với hàm kích hoạt ReLU.

Bảng lượng mực theo lớp cho thấy chữ số 0 tốn nhiều mực nhất với 0,173279 và chữ số 8 đứng thứ
hai với 0,150398 do cả hai đều có vòng kín, còn chữ số 1 ít mực nhất với 0,075961 vì chỉ là một
nét thẳng, thấp hơn chữ số 0 tới 2,28 lần. Sự khác biệt này là một tín hiệu phân biệt thô
mà ngay cả mô hình tuyến tính cũng khai thác được, và nó góp phần lý giải vì sao MNIST đạt độ
chính xác cao ngay cả với kiến trúc rất nhỏ.


## 6. Vì sao MNIST được coi là một mốc chuẩn "dễ"

Báo cáo tổng hợp bằng chứng thu được ở các mục trên thành năm lý do cụ thể.

**Thứ nhất, đối tượng đã được căn giữa và chuẩn hóa kích thước.** Quy trình dựng MNIST áp
dụng phép co ảnh về hộp $20 \times 20$ giữ nguyên tỉ lệ rồi tịnh tiến sao cho trọng tâm khối
mực trùng tâm khung $28 \times 28$. Hệ quả là mạng không phải học tính bất biến tịnh tiến ở
biên độ lớn. Một tầng tích chập với trường tiếp nhận vừa phải đã đủ phủ toàn bộ đối tượng.

**Thứ hai, mỗi ảnh chỉ chứa đúng một đối tượng.** Không có bài toán phát hiện, không có che
khuất, không có nhiều thể hiện của cùng một lớp trong một khung hình. Bài toán rút gọn thành
phân loại toàn ảnh thuần túy.

**Thứ ba, nền đồng nhất và gần như trống.** Hơn 80% điểm ảnh bằng 0 tuyệt đối, tỉ lệ tín hiệu
trên nhiễu rất cao. Bộ lọc tích chập chỉ cần phản ứng với các nét bút, không phải học cách bỏ
qua phông nền phức tạp như ở CIFAR-10 hay ImageNet.

**Thứ tư, dữ liệu là ảnh đơn kênh thang xám.** Không tồn tại biến thiên màu sắc, chiếu sáng
hay cân bằng trắng. Số chiều đầu vào chỉ là $28 \times 28 \times 1 = 784$, nhỏ hơn một bậc
độ lớn so với $32 \times 32 \times 3 = 3\,072$ của CIFAR-10.

**Thứ năm, lớp gần cân bằng và train cùng phân phối với test.** Tỉ số mất cân bằng 1,2437 là
không đáng kể, và hai tập được rút từ cùng nguồn biểu mẫu nên không có dịch chuyển miền.

Hệ quả thực tiễn cho báo cáo này: một mạng tích chập hai tầng cài bằng NumPy thuần đã được kỳ
vọng vượt 97% độ chính xác, còn mạng sâu hơn dùng PyTorch hoặc Keras có Batch Normalization và
Dropout được kỳ vọng tiệm cận ngưỡng 99%. Khoảng cách giữa các cách cài đặt vì thế sẽ hẹp,
và ý nghĩa khoa học của miền MNIST không nằm ở việc đua điểm số mà nằm ở chỗ nó cho phép
**kiểm chứng tính đúng đắn của phép lan truyền ngược viết tay** bằng kiểm tra sai phân hữu hạn,
trước khi chuyển sang miền khó hơn là CIFAR-10.


## 7. Tóm tắt các hằng số chuyển giao cho notebook sau

In [8]:

summary = {
    'n_train_raw': int(N_TRAIN_RAW),
    'n_test_raw': int(N_TEST_RAW),
    'image_shape': [int(IMG_H), int(IMG_W), 1],
    'n_classes': int(N_CLASSES),
    'n_train_split': int(len(idx_tr)),
    'n_val_split': int(len(idx_va)),
    'train_counts': train_counts.tolist(),
    'test_counts': test_counts.tolist(),
    'MEAN': MEAN,
    'STD': STD,
    'random_seed': RANDOM_SEED,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print()
print('Hai hình bắt buộc đã sinh:')
for f in ['fig_mnist_class_distribution.png', 'fig_mnist_sample_grid.png']:
    p = os.path.join(FIG_DIR, f)
    print(f'  {f:42s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')

{
  "n_train_raw": 60000,
  "n_test_raw": 10000,
  "image_shape": [
    28,
    28,
    1
  ],
  "n_classes": 10,
  "n_train_split": 48000,
  "n_val_split": 12000,
  "train_counts": [
    5923,
    6742,
    5958,
    6131,
    5842,
    5421,
    5918,
    6265,
    5851,
    5949
  ],
  "test_counts": [
    980,
    1135,
    1032,
    1010,
    982,
    892,
    958,
    1028,
    974,
    1009
  ],
  "MEAN": 0.1307886838912964,
  "STD": 0.3082403540611267,
  "random_seed": 42
}

Hai hình bắt buộc đã sinh:
  fig_mnist_class_distribution.png               85.2 KB  tồn tại=True
  fig_mnist_sample_grid.png                      97.6 KB  tồn tại=True



**Kết luận notebook 00.** Dữ liệu MNIST đã được xác nhận nguyên vẹn với 60 000 ảnh train và
10 000 ảnh test, mười lớp gần cân bằng, ảnh thang xám $28 \times 28$. Hai hằng số chuẩn hóa
$\mu = 0{,}130789$ và $\sigma = 0{,}308240$ được chốt từ 48 000 ảnh nhánh train và sẽ dùng thống
nhất cho toàn miền. Notebook 01 tiếp nối bằng việc xây dựng mạng tích chập hai chiều hoàn toàn
bằng NumPy, bao gồm suy dẫn toán học từng tầng, kiểm tra gradient bằng sai phân hữu hạn và so
sánh hai biến thể Baseline với Improved.